# 淘宝用户购物行为
## 概述
本数据集包含了2017年11月25日至2017年12月3日之间，有行为的约一百万随机用户的所有行为（行为包括点击、购买、加购、喜欢）。数据集的组织形式和MovieLens-20M类似，即数据集的每一行表示一条用户行为，由用户ID、商品ID、商品类目ID、行为类型和时间戳组成，并以逗号分隔。
## 介绍
用户ID	整数类型，序列化后的用户ID<br />
商品ID	整数类型，序列化后的商品ID<br />
商品类目ID	整数类型，序列化后的商品所属类目ID<br />
行为类型	字符串，枚举类型，包括('pv', 'buy', 'cart', 'fav')<br />
时间戳	行为发生的时间戳<br />
<br />
pv	商品详情页pv，等价于点击<br />
buy	商品购买<br />
cart 将商品加入购物车<br />
fav	收藏商品<br />
## 数据量
维度	数量<br />
用户数量	987,994<br />
商品数量	4,162,024<br />
用户数量	987,994<br />
商品类目数量	9,439<br />
所有行为数量	100,150,807<br />
## 引用
- 1.Han Z, Xiang L, Pengye Z, et al. 2018. Learning Tree-based Deep Model for Recommender Systems. In Proceedings of the 24th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining.
- 2.Han Z, Daqing C, Ziru X, et al. 2019. Joint Optimization of Tree-based Index and Deep Model for Recommender Systems. In Advances in Neural Information Processing Systems.
- 3.Jingwei Z, Ziru X, Wei D, et al. 2020. Learning Optimal Tree Models under Beam Search. In International Conference on Machine Learning.

---

# 解决问题
分析宏观数据。<br/>
将商品依照热度和转化诊断模型进行分类，寻找现象型商品、潜力型商品、流量型商品、滞销型商品。<br />
将用户依照AIDA矩阵进行分类，寻找高价值客户。通过BG/NBD模型判断各类型用户总价值。<br />

---

# 初始化与数据导入

In [19]:
#导入库和文件
import sqlite3 as sql3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import duckdb
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

file_name = 'UserBehavior.csv'
db = 'UserBehavior.db'

In [52]:
#创建数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE OR REPLACE TABLE Raw AS 
    SELECT 
        column0 AS user_id,
        column1 AS good_id,
        column2 AS prop_id,
        column3 AS act,
        column4 AS ts
    FROM read_csv('UserBehavior.csv');
""")

_.close()
print('数据库创建成功！')

数据库创建成功！


In [53]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Raw LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Raw;').fetchdf())
_.close()

   user_id  good_id  prop_id act          ts
0        1  2268318  2520377  pv  1511544070
1        1  2333346  2520771  pv  1511561733
2        1  2576651   149192  pv  1511572885
3        1  3830808  4181361  pv  1511593493
4        1  4365585  2520377  pv  1511596146


---

# 数据清洗

因源文件数据量过大，将database数据导入到dataframe再使用pandas进行数据清洗效率将十分低下，故选择直接在database中进行数据清洗。

In [69]:
_ = duckdb.connect(db)

#清除重复值
_.execute("""
    CREATE OR REPLACE TABLE Cleaned AS 
    SELECT DISTINCT * FROM Raw
""")

#清除缺失值
_.execute("""
    DELETE FROM Cleaned
    WHERE user_id IS NULL 
    OR good_id IS NULL 
    OR act IS NULL
    OR ts IS NULL
    OR prop_id IS NULL;
""")

#清除逻辑异常值
_.execute("""
    DELETE FROM Cleaned
    WHERE act NOT IN ('pv','cart','fav','buy')
    OR (ts < 1511539200 OR ts > 1512230400);
""")

#清理不相关用户
_.execute("""
    DELETE FROM Cleaned
    WHERE user_id NOT IN (SELECT DISTINCT user_id FROM Cleaned WHERE act='pv')
""")

print('数据清洗完成！')
_.close()

数据清洗完成！


In [70]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Cleaned LIMIT 5;").fetchdf())
#print(_.execute('SUMMARIZE Cleaned;').fetchdf())
_.close()

   user_id  good_id  prop_id act          ts
0   112584  1784140  4966058  pv  1512095481
1   112587  3201062   982926  pv  1511664480
2   112587  1599398  1320293  pv  1511664718
3   112587  1625450  2920476  pv  1511667015
4   112597  1185634  4217906  pv  1511959031


---

# 数据格式化

In [76]:
_ = duckdb.connect(db)
#摘取时间特征信息辅助进行根据时间序列的数据分析
_.execute("SET TimeZone = 'Asia/Shanghai';")
_.execute("""
CREATE OR REPLACE VIEW vCleaned AS 
WITH _ AS (
    SELECT 
        *,
        to_timestamp(ts)::TIMESTAMPTZ AS dt
    FROM Cleaned
)
SELECT 
    user_id,
    good_id,
    prop_id,
    act,
    dt AS 时间,
    EXTRACT(DAY FROM dt) AS 日期,
    EXTRACT(HOUR FROM dt) AS 小时,
    EXTRACT(ISODOW FROM dt) AS 星期
FROM _;
""")
_.close()

In [77]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM vCleaned LIMIT 5;").fetchdf())
_.close()

   user_id  good_id  prop_id act                        时间  日期  小时  星期
0   112584  1784140  4966058  pv 2017-12-01 10:31:21+08:00   1  10   5
1   112587  3201062   982926  pv 2017-11-26 10:48:00+08:00  26  10   7
2   112587  1599398  1320293  pv 2017-11-26 10:51:58+08:00  26  10   7
3   112587  1625450  2920476  pv 2017-11-26 11:30:15+08:00  26  11   7
4   112597  1185634  4217906  pv 2017-11-29 20:37:11+08:00  29  20   3


---

# 宏观数据分析

In [81]:
#构建宏观数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE OR REPLACE VIEW macro AS
        SELECT
            count(CASE WHEN act='pv' THEN user_id END) AS 总浏览次数,
            count(CASE WHEN act='fav' THEN user_id END) AS 总收藏次数,
            count(CASE WHEN act='cart' THEN user_id END) AS 总购物车次数,
            count(CASE WHEN act IN ('fav', 'cart') THEN user_id END) AS 总喜爱次数,
            count(CASE WHEN act='buy' THEN user_id END) AS 总购买次数,
            round(总收藏次数/总浏览次数*100,2) AS 浏览转收藏比,
            round(总购物车次数/总浏览次数*100,2) AS 浏览转购物车比,
            round(总购买次数/总收藏次数*100,2) AS 收藏转购买比,
            round(总购买次数/总购物车次数*100,2) AS 购物车转购买比,
            round(总购买次数/总浏览次数*100,2) AS 浏览转购买比
        FROM Cleaned
""")

_.close()

In [82]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM macro LIMIT 5;").fetchdf())
_.close()

      总浏览次数    总收藏次数   总购物车次数    总喜爱次数    总购买次数  浏览转收藏比  浏览转购物车比  收藏转购买比  \
0  77423573  2462929  4719357  7182286  1745425    3.18      6.1   70.87   

   购物车转购买比  浏览转购买比  
0    36.98    2.25  


In [56]:
#读取绘制图所需数据
_ = duckdb.connect(db)
df = _.execute("SELECT 总浏览次数,总喜爱次数,总购买次数 FROM macro").fetchdf()
_.close()

In [57]:
#绘制图
fig = go.Figure(go.Funnel(
    y = list(df.columns),
    x = df.iloc[0],
    textinfo = "value+percent total",
    marker_color=['blue','red','orange'],
    opacity = 0.6))
fig.show()

根据数据图可以观察出，宏观的转化漏斗呈现明显的帕累托分布。仅有10%的浏览次数被转化为喜爱次数，而又有25%的喜爱次数被转化为购买次数。<br/>
我们拥有近90%，接近7000万的浏览次数是无效的，是被用户给忽视的。若我们将浏览转喜爱的转化率仅提高1%，那也能带来70万的总喜爱次数和1.75万的总购买次数，也就能提高至少17.5万的营业额（若按最低购买金额为10元）。<br/>
庞大的用户浏览次数让提高浏览转喜爱的转化率率变得至关重要。<br/>
首先，我们要提高商品展示的推送精准性，优化推荐算法，给用户推送他们切实需要的东西（比如用户迫切需要而线下稀缺的商品）而非热销的大宗商品才能让用户更加想要主动收藏。同时为用户精准推送商品也能够提高用户对平台的依赖性，从而培养更多核心用户，提高总浏览次数，一举两得。<br/>
第二，我们需要更多的营销手段。很多用户可能对商品很感兴趣，但看到商品价格而望而却步，此时我们应部分让利给用户，将一部分平台抽成作为优惠券返还给可能感兴趣的用户，通过补贴手段来拉高需求，提高平台的总产值。尽管短期可能会导致部分亏损，但长期来说能够培养更多的忠实客户，相比各类营销手段效率更高！<br/>
第三，为用户建立收藏页面，能够从中查看他们所有收藏的内容，报告自收藏之后的价格变化，显示与其他平台的价格对比，推荐和收藏商品相关的买家秀图片和评论，将收藏功能打造成真正好用，用户喜爱的功能，从而提高用户粘性和用户收藏商品的意向，也能够刺激用户下单从而提高喜爱转购买的转化率。

In [89]:
#构建时间序列数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE OR REPLACE VIEW Tmacro AS
        SELECT
            小时,
            星期,
            count(CASE WHEN act='pv' THEN user_id END) AS 总浏览次数,
            count(CASE WHEN act='fav' THEN user_id END) AS 总收藏次数,
            count(CASE WHEN act='cart' THEN user_id END) AS 总购物车次数,
            count(CASE WHEN act='buy' THEN user_id END) AS 总购买次数
        FROM vCleaned
        GROUP BY 1,2
""")

_.close()

In [90]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Tmacro LIMIT 5;").fetchdf())
_.close()

   小时  星期   总浏览次数  总收藏次数  总购物车次数  总购买次数
0  10   5  467361  15315   30064  13796
1  17   4  425639  13925   25207  10683
2  12   1  454202  14805   26464  13233
3   6   2  132454   4463    8560   2175
4   8   4  309603  10427   18549   6993


In [84]:
#读取绘制图所需数据
_ = duckdb.connect(db)
df = _.execute("""
SELECT 
小时,
SUM(总浏览次数) AS 总浏览次数,
SUM(总收藏次数) AS 总收藏次数,
SUM(总购物车次数) AS 总购物车次数,
SUM(总购买次数) AS 总购买次数
FROM Tmacro GROUP BY 小时""").fetchdf()
_.close()

In [86]:
#按小时的互动直方图（包含浏览次数）
n = df['小时']
fig = go.Figure(data=[
    go.Bar(name='浏览次数', x=n, y=df['总浏览次数']),
    go.Bar(name='收藏次数', x=n, y=df['总收藏次数']),
    go.Bar(name='购物车次数', x=n, y=df['总购物车次数']),
    go.Bar(name='购买次数', x=n, y=df['总购买次数'])
])
fig.update_layout(barmode='stack',title='<b>按小时的互动直方图（包含浏览次数）</b>')
fig.show()
#按小时的互动直方图（不包含浏览次数）
n = df['小时']
fig = go.Figure(data=[
    go.Bar(name='收藏次数', x=n, y=df['总收藏次数']),
    go.Bar(name='购物车次数', x=n, y=df['总购物车次数']),
    go.Bar(name='购买次数', x=n, y=df['总购买次数'])
])
fig.update_layout(barmode='stack',title='<b>按小时的互动直方图（不包含浏览次数）</b>')
fig.show()

In [32]:
#读取绘制图所需数据
_ = duckdb.connect(db)
df = _.execute("""
SELECT 
小时,
SUM(总浏览次数)/SUM(总购买次数) AS 浏览转购买率
FROM Tmacro GROUP BY 小时""").fetchdf()
_.close()

In [40]:
#统计每小时浏览转购买的转化率
n = df['小时']
fig = go.Figure(data=[
    go.Bar(name='转化率', x=n, y=df['浏览转购买率'],marker=dict(color=df['浏览转购买率'], coloraxis="coloraxis")),
])
# 计算某一个转化率的平均值
mean_val = df['浏览转购买率'].mean()

fig.add_hline(
    y=mean_val, 
    line_dash="dash", 
    line_color="grey", 
    annotation_text=f"平均浏览量: {mean_val:.1f}", # 参考线旁的文字说明
    annotation_position="bottom left"
)

fig.update_layout(barmode='stack',title='<b>按小时的浏览转购买的转化率</b>',yaxis=dict(range=[0, 100]))
fig.show()

根据以上数据可以发现，凌晨4点是全天互动活跃度的最低值——因为此时间是大多数人正在睡觉的时间。随后互动数据一路升高直到早上10点到达一个平台期——即大部分人已睡醒并开始一天的工作，一直维持着较高水平直到晚上7点数据开始继续升高，在晚上9点达到顶峰——即大多数人工作或学习结束后休闲的空挡。随后数据一路下滑直到凌晨4点达到最低值。<br/>
但是浏览转购买的转化率却和上述图像完全相反.凌晨5点时转化率达到最高，之后一路下降到早上10点时进入最低值，此后缓慢升高直到凌晨5点又到达顶峰，这可能是由于人们随着清醒时间的增长，决策判断的心理阈值不断降低的缘故。<br/>
两个数据图形成了一个 互动数据量-转化率 之间的“剪刀差”，导致我们平台产生数据上的失衡，可能导致我们错失潜在的收益。<br/>
我们可以从两个角度缓解这个“剪刀差”的问题，一个角度是提高凌晨时用户的浏览量，另一个角度是提高中午至晚上时段用户的转化率。<br/>


---

# 商品价值分析

按照商品汇总其pv,cart,fav,buy的数量创建两个新的VIEW，分别按照商品id和商品类目id汇总分析。<br/>
定义一个新的数据为like，其值为喜爱或购买某商品的用户数。<br/>
清除掉部分访问极少的商品(pv<5%位数)以防止极端值影响结果真实性。<br/>

In [51]:
#查找浏览量的5%位数
_ = duckdb.connect(db)
print(_.execute("SELECT quantile(独立浏览次数,0.05) FROM Good_Funnel").fetchdf())
print(_.execute("SELECT quantile(独立浏览次数,0.05) FROM Prop_Funnel").fetchdf())
_.close()

   pv_cnt
0    2047
   pv_cnt
0    2150


In [54]:
#构建数据库
_ = duckdb.connect(db)

_.execute("""
    CREATE OR REPLACE VIEW Good_Funnel AS
    WITH matric AS
    (
        SELECT
            good_id,
            count(DISTINCT CASE WHEN act = 'pv' THEN user_id END) AS pv_cnt,
            count(DISTINCT CASE WHEN act IN ('fav', 'cart') THEN user_id END) AS like_cnt,
            count(DISTINCT CASE WHEN act = 'buy' THEN user_id END) AS buy_cnt
        FROM
            Cleaned
        GROUP BY
            good_id
        HAVING
            pv_cnt > 2047
    )
    SELECT
        good_id AS 商品id,
        pv_cnt AS 独立浏览次数,
        like_cnt AS 独立喜爱次数,
        buy_cnt AS 独立购买次数,
        round(like_cnt/pv_cnt*100,2) AS 浏览转喜爱率,
        round(buy_cnt/like_cnt*100,2) AS 喜爱转购买率,
        round(buy_cnt/pv_cnt*100,2) AS 浏览转购买率
    FROM
        matric
    ORDER BY 
        浏览转购买率 DESC
""")

_.execute("""
    CREATE OR REPLACE VIEW Prop_Funnel AS
    WITH matric AS
    (
        SELECT
            prop_id,
            count(DISTINCT CASE WHEN act = 'pv' THEN user_id END) AS pv_cnt,
            count(DISTINCT CASE WHEN act IN ('fav', 'cart') THEN user_id END) AS like_cnt,
            count(DISTINCT CASE WHEN act = 'buy' THEN user_id END) AS buy_cnt
        FROM
            Cleaned
        GROUP BY
            prop_id
        HAVING
            pv_cnt > 2150
    )
    SELECT
        prop_id AS 商品类目id,
        pv_cnt AS 独立浏览次数,
        like_cnt AS 独立喜爱次数,
        buy_cnt AS 独立购买次数,
        round(like_cnt/pv_cnt*100,2) AS 浏览转喜爱率,
        round(buy_cnt/like_cnt*100,2) AS 喜爱转购买率,
        round(buy_cnt/pv_cnt*100,2) AS 浏览转购买率
    FROM
        matric
    ORDER BY 
        浏览转购买率 DESC
""")

_.close()

In [55]:
#检查数据
_ = duckdb.connect(db)
print(_.execute("SELECT * FROM Good_Funnel LIMIT 5;").fetchdf())
print(_.execute("SELECT * FROM Prop_Funnel LIMIT 5;").fetchdf())
_.close()

      商品id  独立浏览次数  独立喜爱次数  独立购买次数  浏览转喜爱率  喜爱转购买率  浏览转购买率
0  3964583    3152     621     575   19.70   92.59   18.24
1  3189426    2108     353     372   16.75  105.38   17.65
2  3237415    2081     310     315   14.90  101.61   15.14
3  5062984    2383     357     315   14.98   88.24   13.22
4  5051027    2172     391     279   18.00   71.36   12.85
    商品类目id  独立浏览次数  独立喜爱次数  独立购买次数  浏览转喜爱率  喜爱转购买率  浏览转购买率
0  4392650    2285     718     960   31.42  133.70   42.01
1  1023823    2541     747     874   29.40  117.00   34.40
2    64179    8515    2757    2794   32.38  101.34   32.81
3   772894    3879     859    1236   22.14  143.89   31.86
4    85955    8298    2311    2629   27.85  113.76   31.68


根据商品的浏览次数，浏览转购买率将各类商品划分为四个品类。
|品类名称|浏览量 (X)|转化率 (Y)|购买量(Z)|
|------|------|------|------|
|现象型商品 |高|高|特大|
|流量型商品 |高|低|大|
|潜力型商品 |低|高|大|
|滞销型商品 |低|低|小|

In [ ]:
#计算分界线
_ = duckdb.connect(db)
print(_.execute("SELECT quantile(独立浏览次数,0.5),quantile(浏览转购买率,0.5) FROM Good_Funnel;").fetchdf())
print(_.execute("SELECT quantile(独立浏览次数,0.5),quantile(浏览转购买率,0.5) FROM Prop_Funnel;").fetchdf())
_.close()

In [78]:
#绘制商品类目气泡图
_ = duckdb.connect(db)
df = _.execute("SELECT 独立浏览次数,浏览转购买率,独立购买次数 FROM Prop_Funnel").fetchdf()
_.close()

fig = go.Figure(data=[
    go.Scatter(
        x=df['独立浏览次数'], 
        y=df['浏览转购买率'],
        mode='markers',
        opacity = 0.7,
        marker=dict(size=df['独立购买次数'],sizemode='area',sizeref=50,color=df['独立购买次数'], coloraxis="coloraxis")),
])

mean_x = df['独立浏览次数'].mean()
mean_y = df['浏览转购买率'].mean()

fig.add_vline(
    x=mean_x, 
    line_dash="dash", 
    line_color="grey", 
    annotation_text=f"平均浏览量: {mean_x:.1f}", # 参考线旁的文字说明
    annotation_position="bottom left"
)
fig.add_hline(
    y=mean_y, 
    line_dash="dash", 
    line_color="grey", 
    annotation_text=f"平均转化率: {mean_y:.1f}", # 参考线旁的文字说明
    annotation_position="bottom left"
)

fig.update_layout(title='<b>商品类目象限图</b>')
fig.show()

In [82]:
#绘制商品气泡图
_ = duckdb.connect(db)
df = _.execute("SELECT 独立浏览次数,浏览转购买率,独立购买次数 FROM Good_Funnel").fetchdf()
_.close()

fig = go.Figure(data=[
    go.Scatter(
        x=df['独立浏览次数'], 
        y=df['浏览转购买率'],
        mode='markers',
        opacity = 0.7,
        marker=dict(size=df['独立购买次数'],sizemode='area',sizeref=1,color=df['独立购买次数'], coloraxis="coloraxis")),
])

mean_x = df['独立浏览次数'].mean()
mean_y = df['浏览转购买率'].mean()

fig.add_vline(
    x=mean_x, 
    line_dash="dash", 
    line_color="grey", 
    annotation_text=f"平均浏览量: {mean_x:.1f}", # 参考线旁的文字说明
    annotation_position="bottom left"
)
fig.add_hline(
    y=mean_y, 
    line_dash="dash", 
    line_color="grey", 
    annotation_text=f"平均转化率: {mean_y:.1f}", # 参考线旁的文字说明
    annotation_position="bottom left"
)

fig.update_layout(title='<b>商品象限图</b>')
fig.show()

根据图像可以看出，在我们平台的大多数商品都处于左下角的滞销型商品类目，第二多的商品类型是潜力型商品，第三多的商品类型是流量型商品，最少的是现象型商品。<br/>
对于电商平台，根据长尾效应，保证大量的滞销型商品类目对平台同样至关重要——这80%的商品也能够带来20%的利润。所以相比于花费大成本提高滞销型商品的浏览量和转化率，我们更应该将成本花在提高潜力型商品的浏览量和流量型商品的转化率上。<br/>
对于潜力型商品，最简单的方法是提高其展示频率。我们也可以建议商家更换更吸引用户的封面和标题，让商家主动去创造流量。或将高浏览、低转化的流量型商品与这类高转化的潜力型商品在购物车或结算页进行跨品类组合推荐，利用大盘流量的富余流量带动潜力款。<br/>
对于流量型商品，大多数商品都是依赖商家投钱推广，多为低成本、高利润率、潮流型商品，平台最好的方法是将其投流的流量进行精准投放，将购买的流量更多推广给感兴趣的顾客，同时若流量型商品的转化率持续低于大盘底线，系统应自动收窄自然曝光，倒逼商家优化详情页或下架劣质SKU。低成本潮流品极度依赖冲动消费。平台应在推荐流中开放短视频、买家秀组件或直播切片挂载，用动态场景替代静态图，拉近“浏览”到“下单”的心理距离。<br/>

# 用户价值与行为特征分析